In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
df_2018_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2018-01.csv", sep=";")

In [3]:
df_2018_01.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,GNV,02/01/2018,"2,37","1,7383",R$ / m³,PETROBRAS DISTRIBUIDORA S.A.
1,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,DIESEL S10,02/01/2018,"3,24","3,1366",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,ETANOL,02/01/2018,"2,93","2,6194",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,NE,BA,SALVADOR,PETROBRAS DISTRIBUIDORA S.A.,34.274.233/0015-08,RUA EDISTIO PONDE,474,NaN,STIEP,41770-395,GASOLINA,02/01/2018,"3,62","3,6017",R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
4,S,RS,CANOAS,METROPOLITANO COMERCIO DE COMBUSTIVEIS LTDA,88.587.589/0001-17,AVENIDA GUILHERME SCHELL,6340,NaN,CENTRO,92310-000,GNV,02/01/2018,"2,699",NaN,R$ / m³,BRANCA


In [4]:
df_2018_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2018-02.csv", sep=";")

In [5]:
df_2018 =  pd.concat((df_2018_01, df_2018_02,))

In [6]:
df_2018.info()

<class 'pandas.core.frame.DataFrame'>
Index: 963002 entries, 0 to 493168
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Regiao - Sigla     963002 non-null  object
 1   Estado - Sigla     963002 non-null  object
 2   Municipio          963002 non-null  object
 3   Revenda            963002 non-null  object
 4   CNPJ da Revenda    963002 non-null  object
 5   Nome da Rua        963002 non-null  object
 6   Numero Rua         962208 non-null  object
 7   Complemento        250596 non-null  object
 8   Bairro             960044 non-null  object
 9   Cep                963002 non-null  object
 10  Produto            963002 non-null  object
 11  Data da Coleta     963002 non-null  object
 12  Valor de Venda     963002 non-null  object
 13  Valor de Compra    386490 non-null  object
 14  Unidade de Medida  963002 non-null  object
 15  Bandeira           963002 non-null  object
dtypes: object(16)
memory usag

In [7]:
df_2018_pe = df_2018[df_2018["Estado - Sigla"] == "PE"]

In [8]:
df_2018_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
3296,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,DIESEL S10,02/01/2018,"3,299",NaN,R$ / litro,RAIZEN
3297,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,DIESEL,02/01/2018,"3,299",NaN,R$ / litro,RAIZEN
3298,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,ETANOL,02/01/2018,"3,25",NaN,R$ / litro,RAIZEN
3299,NE,PE,ARARIPINA,JUVENAL ANGELO & CIA LTDA,04.965.564/0003-81,AVENIDA GOVERNADOR MUNIZ FALCAO,525,A,CENTRO,56280-000,GASOLINA,02/01/2018,"4,34",NaN,R$ / litro,RAIZEN
3300,NE,PE,ARARIPINA,POSTO TREVO COMBUSTIVEIS EIRELI,07.920.417/0001-11,AVENIDA FLORENTINO ALVES BATISTA,910,NaN,CENTRO,56280-000,DIESEL S10,02/01/2018,"3,299","3,296",R$ / litro,DISLUB


In [9]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Staging no BigQuery
table_id = os.getenv("BRONZE_2018")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [10]:
# Criação e carga da tabela BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2018_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
